# Two-Dimensional Spatial Convergence

This tutorial refines a two-dimensional square mesh while holding the angular quadrature fixed. Successive values of the leakage through the right boundary are compared to estimate the spatial convergence rate.

## Define the self-convergence metric

The unit square contains a one-group material with $\Sigma_t=1$ and scattering ratio $c=\Sigma_s/\Sigma_t=0.8$. A uniform isotropic source of unit strength is applied throughout the square. The left and right boundaries are vacuum, while the top and bottom boundaries are reflecting.

Scattering removes the simple characteristic reference available for a pure absorber. Instead, let $J_N$ denote the right-boundary leakage calculated using $N\times N$ cells and define the successive-grid difference

$$d_N=\left|J_N-J_{2N}\right|.$$

If the leakage error behaves as $Ch^p$, the observed order follows from three successive meshes:

$$p_N=\log_2\left(\frac{d_N}{d_{2N}}\right).$$

The angular quadrature and iterative-solver tolerance remain fixed so that the measured differences are dominated by spatial refinement.

In [ ]:
import math
from mpi4py import MPI
from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

rank = MPI.COMM_WORLD.rank
quadrature = GLCProductQuadrature2DXY(
    n_polar=2, n_azimuthal=16, scattering_order=0
)

def transmitted_current(num_cells):
    nodes = [i / num_cells for i in range(num_cells + 1)]
    mesh = OrthogonalMeshGenerator(node_sets=[nodes, nodes]).Execute()
    mesh.SetUniformBlockID(0)

    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.8)
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[
            {
                "groups_from_to": (0, 0),
                "angular_quadrature": quadrature,
                "inner_linear_method": "petsc_gmres",
                "l_abs_tol": 1.0e-12,
                "l_max_its": 300,
                "gmres_restart_interval": 100,
            }
        ],
        xs_map=[{"block_ids": [0], "xs": xs}],
        volumetric_sources=[
            VolumetricSource(block_ids=[0], group_strength=[1.0])
        ],
        boundary_conditions=[
            {"name": "xmin", "type": "vacuum"},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymin", "type": "reflecting"},
            {"name": "ymax", "type": "reflecting"},
        ],
        options={
            "save_angular_flux": True,
            "verbose_inner_iterations": False,
        },
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()

    return float(problem.ComputeLeakage(["xmax"])["xmax"][0])

## Refine the mesh

The number of cells in each coordinate direction is doubled at every level. The leakage is an integrated outflow response and approaches third-order superconvergence for this smooth problem. This result does not imply third-order convergence of the flux field throughout the domain.

In [ ]:
resolutions = [8, 16, 32, 64]
leakage_values = [
    transmitted_current(num_cells) for num_cells in resolutions
]
differences = [
    abs(coarse - fine)
    for coarse, fine in zip(leakage_values, leakage_values[1:])
]
orders = [
    math.log(differences[index - 1] / differences[index], 2.0)
    for index in range(1, len(differences))
]

if rank == 0:
    for cells, leakage in zip(resolutions, leakage_values):
        print(f"{cells:3d} x {cells:3d} cells: leakage={leakage:.10e}")
    print(f"Spatial-convergence final order={orders[-1]:.6e}")
    print(
        f"Spatial-convergence final difference={differences[-1]:.6e}"
    )
assert all(
    fine < coarse for coarse, fine in zip(differences, differences[1:])
)
assert orders[-1] > 2.8
assert differences[-1] < 3.0e-7

## Plot the convergence

The successive-grid leakage difference is plotted against the coarser resolution in each comparison. The dashed $N^{-3}$ line illustrates the asymptotic behavior.

In [ ]:
import matplotlib.pyplot as plt

if rank == 0:
    comparison_resolutions = resolutions[:-1]
    third_order = [
        differences[0] * (comparison_resolutions[0] / num_cells) ** 3
        for num_cells in comparison_resolutions
    ]

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.loglog(
        comparison_resolutions,
        differences,
        "o-",
        label=r"$|J_N-J_{2N}|$",
    )
    ax.loglog(
        comparison_resolutions,
        third_order,
        "--",
        label=r"$N^{-3}$ reference",
    )
    ax.set_xlabel("Cells per coordinate direction, $N$")
    ax.set_ylabel("Successive-grid leakage difference")
    ax.set_title("Leakage self-convergence with reflecting boundaries")
    ax.grid(True, which="both", linestyle=":", alpha=0.6)
    ax.legend()
    fig.tight_layout()
    # fig.savefig("images/spatial_convergence.png", dpi=200, bbox_inches="tight")

if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()

![Log-log plot of the successive-grid right-boundary leakage difference and third-order reference curve.](./images/spatial_convergence.png)